In [0]:
Use Catalog sql;

1. Return the latest order for each customer

In [0]:
-- CREATE SCHEMA day_1;

CREATE or REPLACE TABLE sql.day_1.orders (
    customer_id VARCHAR(10),
    order_date DATE,
    amount INT
);

INSERT INTO sql.day_1.orders (customer_id, order_date, amount) VALUES
('C001', '2024-01-10', 250),
('C001', '2024-03-22', 800),
('C001', '2024-07-15', 150),
('C002', '2024-02-05', 400),
('C002', '2024-06-18', 950),
('C003', '2024-04-01', 300);

SELECT * FROM sql.day_1.orders;

In [0]:
-- Return the latest order for each customer:
--1. Window Function:
select customer_id, order_date, amount from (
    select customer_id, order_date, amount,
    rank() over (partition by customer_id order by order_date desc) as rnk
    from sql.day_1.orders
) where rnk=1;

--1. Using Max:
select o.customer_id, o.order_date, o.amount 
from sql.day_1.orders o 
join (
    select customer_id, max(order_date) as latest_order_date
    from sql.day_1.orders group by customer_id

) m 
on o.customer_id = m.customer_id and o.order_date = m.latest_order_date

22. For each day, sum the readings taken at ODD positions and EVEN positions separately.

In [0]:
-- CREATE SCHEMA day_22;
CREATE or REPLACE TABLE sql.day_22.heart_rate_log (
    log_id INT,
    reading_time TIMESTAMP,
    bpm INT
);

INSERT INTO sql.day_22.heart_rate_log (log_id, reading_time, bpm) VALUES
(101, '2024-03-10 06:00:00', 62),
(102, '2024-03-10 09:30:00', 88),
(103, '2024-03-10 13:15:00', 75),
(104, '2024-03-10 18:45:00', 110),
(105, '2024-03-10 22:00:00', 68),
(106, '2024-03-11 07:00:00', 70),
(107, '2024-03-11 12:30:00', 95),
(108, '2024-03-11 19:00:00', 120),
(109, '2024-03-11 23:00:00', 65);

select * from sql.day_22.heart_rate_log;

--Output
--    day	           odd_pos_sum	    even_pos_sum
-- 2024-03-10	        205	            198
-- 2024-03-11	        190	            160

In [0]:
-- For each day, sum the readings taken at ODD positions and EVEN positions separately.
with cte1 as (
    select *, row_number() over (partition by date(reading_time) order by reading_time asc) as row_nm
    from sql.day_22.heart_rate_log
)
select date(reading_time),
sum(case when row_nm % 2 != 0 then bpm end) as odd_pos_sum,
sum(case when row_nm % 2 == 0 then bpm end) as even_pos_sum
from cte1 group by 1;